In [1]:
from time import time
import pandas as pd
import numpy as np
from collections import OrderedDict
import warnings

import pandas as pd
from CBFV.composition import generate_features

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression

from sklearn.ensemble import (
    AdaBoostClassifier, GradientBoostingClassifier,
    RandomForestClassifier, ExtraTreesClassifier
)

from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC, LinearSVC

from sklearn.preprocessing import StandardScaler

In [2]:
def instantiate_model(model_name):
    model = model_name()
    return model

def fit_model(model, X_train, y_train):
    ti = time()
    model = instantiate_model(model)
    model.fit(X_train, y_train)
    fit_time = time() - ti
    return model, fit_time

def append_result_df(df, result_dict):
    df_result_appended = pd.concat([df, pd.DataFrame([result_dict])], ignore_index=True)
    return df_result_appended

def append_model_dict(dic, model_name, model):
    dic[model_name] = model
    return dic

def evaluate_model(model, X, y_act):
    y_pred = model.predict(X)
    acc = accuracy_score(y_act, y_pred)
    f1 = f1_score(y_act, y_pred, average="weighted", zero_division=0)
    precision = precision_score(y_act, y_pred, average="weighted", zero_division=0)
    recall = recall_score(y_act, y_pred, average="weighted", zero_division=0)
    return acc, f1, precision, recall

def fit_evaluate_model(model, model_name, X_train, y_train, X_val, y_val):
    model, fit_time = fit_model(model, X_train, y_train)
    acc_train, f1_train, prec_train, rec_train = evaluate_model(model, X_train, y_train)
    acc_val, f1_val, prec_val, rec_val = evaluate_model(model, X_val, y_val)
    result_dict = {
        'model_name': model_name,
        'model_name_pretty': type(model).__name__,
        'model_params': model.get_params(),
        'fit_time': fit_time,
        'acc_train': acc_train,
        'f1_train': f1_train,
        'precision_train': prec_train,
        'recall_train': rec_train,
        'acc_val': acc_val,
        'f1_val': f1_val,
        'precision_val': prec_val,
        'recall_val': rec_val,
    }
    return model, result_dict

# Reading in dataset

In [3]:
df_train = pd.read_csv('control_dataset_splits/ICSD_train_split.csv')
df_test = pd.read_csv('control_dataset_splits/ICSD_test_split.csv')
df_val = pd.read_csv('control_dataset_splits/ICSD_val_split.csv')

In [4]:
df_train["Crystal_System"] = df_train["Crystal_System"].astype("category")
df_test["Crystal_System"] = df_test["Crystal_System"].astype("category")
df_val["Crystal_System"] = df_val["Crystal_System"].astype("category")


df_train["target"] = df_train["Crystal_System"].cat.codes
df_test["target"] = df_test["Crystal_System"].cat.codes
df_val["target"] = df_val["Crystal_System"].cat.codes

label_mapping = dict(enumerate(df_train["Crystal_System"].cat.categories))

In [5]:
rename_dict = {'Chemical': 'formula'}

df_train = df_train.rename(columns=rename_dict)
df_val = df_val.rename(columns=rename_dict)
df_test = df_test.rename(columns=rename_dict)

In [6]:
# CBFV needs 'formula' and 'target' columns only
df_train_cbfv = df_train[['formula', 'target', 'Temperature', 'Pressure']].copy()
df_val_cbfv   = df_val[['formula', 'target', 'Temperature', 'Pressure']].copy()
df_test_cbfv  = df_test[['formula', 'target', 'Temperature', 'Pressure']].copy()

# Generate Oliynyk  features

In [7]:
X_train_unscaled, y_train, formulae_train, skipped_train = generate_features(
    df_train_cbfv, elem_prop='oliynyk', drop_duplicates=False, extend_features=True, sum_feat=True)
X_val_unscaled, y_val, formulae_val, skipped_val = generate_features(
    df_val_cbfv, elem_prop='oliynyk', drop_duplicates=False, extend_features=True, sum_feat=True)
X_test_unscaled, y_test, formulae_test, skipped_test = generate_features(
    df_test_cbfv, elem_prop='oliynyk', drop_duplicates=False, extend_features=True, sum_feat=True)

Processing Input Data: 100%|██████████| 1617/1617 [00:00<00:00, 54211.55it/s]


	Featurizing Compositions...


Assigning Features...: 100%|██████████| 1617/1617 [00:00<00:00, 45842.33it/s]


	Creating Pandas Objects...


Processing Input Data: 100%|██████████| 438/438 [00:00<00:00, 75507.82it/s]


...

	Creating Pandas Objects...


Processing Input Data: 100%|██████████| 228/228 [00:00<00:00, 65504.58it/s]


	Featurizing Compositions...


Assigning Features...: 100%|██████████| 228/228 [00:00<00:00, 40890.29it/s]


	Creating Pandas Objects...


In [8]:
print(f"Skipped train: {len(skipped_train)}")
print(f"Skipped val:   {len(skipped_val)}")
print(f"Skipped test:  {len(skipped_test)}")

Skipped train: 0
Skipped val:   0
Skipped test:  0


In [9]:
feature_cols = [col for col in X_train_unscaled.columns if col not in ['HMS', 'Crystal_System', 'target']]
print(feature_cols)
print(X_train_unscaled.shape)  # should be (n_samples, n_features)
print(y_train.shape)  # should be (n_samples,)
print(label_mapping)

['sum_Atomic_Number', 'sum_Atomic_Weight', 'sum_Period', 'sum_group', 'sum_families', 'sum_Metal', 'sum_Nonmetal', 'sum_Metalliod', 'sum_Mendeleev_Number', 'sum_l_quantum_number', 'sum_Atomic_Radius', 'sum_Miracle_Radius_[pm]', 'sum_Covalent_Radius', 'sum_Zunger_radii_sum', 'sum_ionic_radius', 'sum_crystal_radius', 'sum_Pauling_Electronegativity', 'sum_MB_electonegativity', 'sum_Gordy_electonegativity', 'sum_Mulliken_EN', 'sum_Allred-Rockow_electronegativity', 'sum_metallic_valence', 'sum_number_of_valence_electrons', 'sum_gilmor_number_of_valence_electron', 'sum_valence_s', 'sum_valence_p', 'sum_valence_d', 'sum_valence_f', 'sum_Number_of_unfilled_s_valence_electrons', 'sum_Number_of_unfilled_p_valence_electrons', 'sum_Number_of_unfilled_d_valence_electrons', 'sum_Number_of_unfilled_f_valence_electrons', 'sum_outer_shell_electrons', 'sum_1st_ionization_potential_(kJ/mol)', 'sum_polarizability(A^3)', 'sum_Melting_point_(K)', 'sum_Boiling_Point_(K)', 'sum_Density_(g/mL)', 'sum_specific_

In [10]:
from sklearn.preprocessing import normalize
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_unscaled)  # fit only on train
X_val   = scaler.transform(X_val_unscaled)
X_test  = scaler.transform(X_test_unscaled)

In [11]:
df_classics = pd.DataFrame(columns=[
    'model_name',
    'model_name_pretty',
    'model_params',
    'fit_time',
    'acc_train',
    'f1_train',
    'precision_train',
    'recall_train',
    'acc_val',
    'f1_val',
    'precision_val',
    'recall_val'
])

RANDOM_SEED = 42

classic_model_classes = OrderedDict({
    'dumc': DummyClassifier,
    'lr':   LogisticRegression,
    'abc':  AdaBoostClassifier,
    'gbc':  GradientBoostingClassifier,
    'rfc':  RandomForestClassifier,
    'etc':  ExtraTreesClassifier,
    'svc':  SVC,
    'lsvc': LinearSVC,
    'knc':  KNeighborsClassifier,
})

# Models with seeds and convergence fixes applied
classic_model_names = OrderedDict({
    'dumc': lambda: DummyClassifier(random_state=RANDOM_SEED),
    'lr':   lambda: LogisticRegression(random_state=RANDOM_SEED, max_iter=1000),
    'abc':  lambda: AdaBoostClassifier(random_state=RANDOM_SEED),
    'gbc':  lambda: GradientBoostingClassifier(random_state=RANDOM_SEED),
    'rfc':  lambda: RandomForestClassifier(random_state=RANDOM_SEED),
    'etc':  lambda: ExtraTreesClassifier(random_state=RANDOM_SEED),
    'svc':  lambda: SVC(random_state=RANDOM_SEED),
    'lsvc': lambda: LinearSVC(random_state=RANDOM_SEED, max_iter=5000),
    'knc':  lambda: KNeighborsClassifier(),  # no random_state needed
})

df_classics = pd.DataFrame()
classic_models = OrderedDict()

ti = time()
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    for model_name, model in classic_model_names.items():
        print(f'Now fitting and evaluating model {model_name}: {model}')
        model, result_dict = fit_evaluate_model(
            model, model_name, X_train, y_train, X_val, y_val
        )
        df_classics = append_result_df(df_classics, result_dict)
        classic_models = append_model_dict(classic_models, model_name, model)

dt = time() - ti
print(f'Finished fitting {len(classic_models)} models, total time: {dt:0.2f} s')

Now fitting and evaluating model dumc: <function <lambda> at 0x7acd417ad440>
Now fitting and evaluating model lr: <function <lambda> at 0x7acd417aef20>
Now fitting and evaluating model abc: <function <lambda> at 0x7acd417aed40>
Now fitting and evaluating model gbc: <function <lambda> at 0x7acd417aee80>
Now fitting and evaluating model rfc: <function <lambda> at 0x7acd417ae8e0>
Now fitting and evaluating model etc: <function <lambda> at 0x7acd417af060>
Now fitting and evaluating model svc: <function <lambda> at 0x7acd417af100>
Now fitting and evaluating model lsvc: <function <lambda> at 0x7acd417af1a0>
Now fitting and evaluating model knc: <function <lambda> at 0x7acd417af240>
Finished fitting 9 models, total time: 80.68 s


In [12]:
# Sort in order of increasing validation r2 score
df_classics = df_classics.sort_values('acc_val', ignore_index=True)
df_classics

,model_name,model_name_pretty,model_params,fit_time,acc_train,f1_train,precision_train,recall_train,acc_val,f1_val,precision_val,recall_val
0,dumc,DummyClassifier,"{'constant': None, 'random_state': 42, 'strate...",0.000926,0.313544,0.149686,0.098310,0.313544,0.273973,0.117838,0.075061,0.273973
1,abc,AdaBoostClassifier,"{'algorithm': 'deprecated', 'estimator': None,...",1.214046,0.594310,0.575872,0.582430,0.594310,0.534247,0.501536,0.518546,0.534247
2,lr,LogisticRegression,"{'C': 1.0, 'class_weight': None, 'dual': False...",16.949571,0.820656,0.819248,0.820111,0.820656,0.700913,0.694409,0.697472,0.700913
3,svc,SVC,"{'C': 1.0, 'break_ties': False, 'cache_size': ...",0.207047,0.788497,0.786031,0.789559,0.788497,0.703196,0.692623,0.698728,0.703196
4,lsvc,LinearSVC,"{'C': 1.0, 'class_weight': None, 'dual': 'auto...",27.212221,0.828077,0.826196,0.828012,0.828077,0.716895,0.712551,0.719217,0.716895
5,knc,KNeighborsClassifier,"{'algorithm': 'auto', 'leaf_size': 30, 'metric...",0.001669,0.816945,0.816328,0.817750,0.816945,0.739726,0.736572,0.736949,0.739726
6,gbc,GradientBoostingClassifier,"{'ccp_alpha': 0.0, 'criterion': 'friedman_mse'...",32.540944,0.973408,0.973428,0.973738,0.973408,0.794521,0.795690,0.801943,0.794521
7,etc,ExtraTreesClassifier,"{'bootstrap': False, 'ccp_alpha': 0.0, 'class_...",0.420208,0.987631,0.987617,0.987822,0.987631,0.842466,0.841800,0.843528,0.842466
8,rfc,RandomForestClassifier,"{'bootstrap': True, 'ccp_alpha': 0.0, 'class_w...",1.204834,0.987631,0.987624,0.987736,0.987631,0.842466,0.842192,0.844020,0.842466


In [13]:
from tqdm import tqdm
# Combine Unscaled train and val together
X_train_final = np.concatenate((X_train_unscaled, X_val_unscaled), axis=0)
y_train_final = np.concatenate((y_train, y_val), axis=0)
# Scale on the train_final and fit the test

scaler = StandardScaler()
X_train_final = scaler.fit_transform(X_train_final)  # fit only on train
X_test  = scaler.transform(X_test_unscaled.to_numpy())

seeds = [42, 123, 456, 789, 1024, 2024, 314, 99, 7, 2000]
models_to_evaluate = ['etc', 'rfc', 'lr', 'lsvc', 'dumc']
seed_rows = []

for seed in tqdm(seeds, desc='Seeds'):
    for model_name in tqdm(models_to_evaluate, desc='Models', leave=False):
        
        model_params = df_classics.loc[df_classics['model_name'] == model_name].iloc[0]['model_params'].copy()
        model_params['random_state'] = seed
        
        model = classic_model_classes[model_name](**model_params)
        model.fit(X_train_final, y_train_final)
        
        acc, f1, precision, recall = evaluate_model(model, X_test, y_test)
        
        seed_rows.append({
            'seed':      seed,
            'model':     model_name,
            'test_acc':  acc,
            'test_f1':   f1,
            'test_prec': precision,
            'test_rec':  recall,
        })

seed_df = pd.DataFrame(seed_rows)

summary = seed_df.groupby('model').agg(
    acc_mean=('test_acc', 'mean'),
    acc_std= ('test_acc', 'std'),
    f1_mean= ('test_f1',  'mean'),
    f1_std=  ('test_f1',  'std'),
).round(4)

print(summary)

seed_df['feature_set'] = 'oliynyk' 
seed_df.to_csv('results/oliynyk_seed_results.csv', index=False)

Seeds: 100%|██████████| 10/10 [09:16<00:00, 55.66s/it]

       acc_mean  acc_std  f1_mean  f1_std
model                                    
dumc     0.3026   0.0000   0.1406  0.0000
etc      0.8605   0.0085   0.8625  0.0083
lr       0.6623   0.0000   0.6585  0.0000
lsvc     0.7061   0.0000   0.7020  0.0000
rfc      0.8539   0.0083   0.8560  0.0081
